<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>7. Operaciones sobre elementos: transformación.</i></font>  </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

In [ ]:
import pandas as pd
import numpy as np

pd.__version__

Para esta libreta se trabajará con una versión del `DataFrame` de prueba `df_productos`, utilizado en libretas anteriores, a la que se añadieron algunas columnas.

In [ ]:
df_productos = pd.read_csv('data/productos_prep.csv', index_col=0, dtype={'C.P':str})
df_productos

In [ ]:
df_productos.dtypes

Para ilustrar algunas de las operaciones, se utilizará una serie, `s_ventas`, obtenida a partir de la columna `df_productos['Venta']`.

In [ ]:
s_ventas = df_productos['Venta']
s_ventas

---
## <font color="#5E5E5E"> 1. Operaciones sobre `Series`: vectorización</font>

La *vectorización*  consiste en aplicar operaciones sobre colecciones de datos, y que esas operaciones se apliquen de manera individual, y en paralelo, a los datos.  Los objetos de tipo `Series` soportan vectorización. Así, cualquier operación aritmética que implica a un objeto `Series` se hace sobre cada uno de los elementos. El resultado es otro objeto de tipo `Series`.

En la siguiente celda, se suma uno a cada elemento de la serie `s_ventas`.

In [ ]:
s_ventas + 1

Puesto que las columnas de un `DataFrame` son `Series`, se puede operar directamente con ellas.

In [ ]:
df_productos['Venta'] + 1

También se pueden construir expresiones vectorizadas que impliquen a varias  columnas (varias `Series`). 

In [ ]:
df_productos['Venta']-df_productos['Compra']

En muchos casos, el resultado se utiliza para crear una nueva columna.

In [ ]:
df_productos['Margen']=df_productos['Venta']-df_productos['Compra']
df_productos

<div class="alert alert-block alert-info">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
__Importante__: Al operar entre dos `Series`, pandas alinea los valores por las etiquetas de sus índices, no por su posición. Si una etiqueta solo aparece en una de ellas, el resultado de esa etiqueta es un valor perdido. Si una etiqueta está repetida, el resultado depende también de si los índices completos coinciden: **con índices idénticos se opera posición a posición**; al alinear índices distintos pueden aparecer todas las combinaciones de las filas con esa etiqueta.
</div>

In [ ]:
s_ventas

In [ ]:
s_compras = df_productos['Compra']
s_compras

In [ ]:
s_compras+s_ventas

In [ ]:
s_compras = df_productos['Compra'].sample(frac=1) # Desordena la columna y la copia
s_compras

In [ ]:
s_compras+s_ventas

In [ ]:
s_compras['Tienda 5'] = 100 
s_compras

In [ ]:
s_compras + s_ventas

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
    En la suma anterior se observa que aparecen cuatro entradas correspondientes a `Tienda 2`. Esto se debe a que hay dos entradas en cada una de las series, y se forman las cuatro parejas posibles entre las dos apariciones de cada serie. *Es muy importante *tener presente este comportamiento**.
</div>


La clase `Series` implementa estos y otros operadores como métodos [**( + inf )**](https://pandas.pydata.org/pandas-docs/stable/reference/series.html#binary-operator-functions).

In [ ]:
df_productos['Venta'].floordiv(2)  # División entera (no tiene por qué devolver un entero)

Los `NaN` se propagan en las operaciones. Sin embargo, los métodos *built-in* para operaciones aritméticas, permiten sustituir los valores `NaN` en las series operandos mediante el parámetro `fill_value`.

In [ ]:
s_ventas.sub(s_compras)
#s_ventas.sub(s_compras, fill_value=1000)

Por otra parte, también es posible aplicar funciones *universales*  [**( + inf )**](https://numpy.org/doc/stable/reference/ufuncs.html#available-ufuncs) de *NumPy*, de manera vectorizada, a los elementos de una `Series`.

In [ ]:
np.floor_divide(df_productos['Venta'], 2)                          # División entera
# np.floor_divide(df_productos['Venta'], df_productos['Compra'])

Es posible definir funciones que reciban como argumento una `Series` y actúen sobre todos sus elementos, *también* de manera vectorizada. Para ello, las operaciones que efectúan deben serlo también.

In [ ]:
def division(s,d):
    return s//d

division(df_productos['Venta'],2)

Estas funciones pueden actuar operando con varias `Series`, ya que es equivalente a hacer la operación y devolver la `Serie` con el resultado.

In [ ]:
def margen(s_ventas, s_compra):
    return s_ventas - s_compra

margen(df_productos['Venta'],df_productos['Compra'])

`DataFrame.assign()` devuelve un nuevo `DataFrame` con las columnas añadidas o actualizadas. El objeto original no se modifica si no guardamos el resultado sobre él.

In [ ]:
df_productos.assign(Calle=pd.Series({'Tienda 1': 'Ramón y Cajal', 'Tienda 2': 'Hellín'}))
#df_productos.assign(VENTA_IVA=df_productos['Venta'] * 1.21) 

`DataFrame.assign()` puede recibir como argumento una función que opere sobre el propio `DataFrame`. 

In [ ]:
df_productos.assign(VENTA_IVA=lambda df: df.Venta * 1.21)

Desde la versión 3.0.0 de pandas se puede utilizar la nueva expresión `pd.col()` que hace referencia a una columna del `DataFrame`.

In [ ]:
df_productos.assign(VENTA_IVA=pd.col('Venta') * 1.21)  # pandas 3.0

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> Este uso de `DataFrame.assign()` es interesante, sobre todo, cuando se hace encadenamiento de métodos (se verá después).
</div>

---
## <font color="#5E5E5E"> 2. Operaciones vectorizadas sobre `Series` de `String`: `Series.str`</font>



Algunas operaciones, como la concatenación,  se pueden aplicar sobre `Series` de `strings` también de manera vectorizada.

In [ ]:
df_productos['Nombre']+ " " +df_productos['Apellido']   # Concatena nombres y apellidos.

_pandas_ implementa un conjunto de operaciones para el manejo de `Series` de `strings`, y que se aplican a nivel de elemento de forma vectorizada. Muchas de estas funciones corresponden con las equivalentes en las librerías de _Python_, y otras son versiones extendidas (por ejemplo, trabajan expresiones regulares). El listado completo de métodos puede encontrarse en la sección correspondiente de la API [**(+ inf)**](https://pandas.pydata.org/docs/user_guide/text.html#method-summary). 

A través del objeto "`accessor`" `Series.str` se puede acceder a estos métodos.

In [ ]:
df_productos['Producto'].str.upper()                                 # Pasa a mayúsculas
#df_productos['Producto'].str.replace(r'\s\w+','', regex=True)       # Busca un patrón (expresión regular) y lo sustituye

Algunos son muy útiles. Por ejemplo, el método `Series.str.split()` divide un `string` y devuelve una lista con las partes. 

In [ ]:
df_productos['Producto'].str.split(' ') # Separa con un espacio
#df_productos['Producto'].str.split()    # Separa con uno o varios espacios, tabuladores. 

#df_productos['Producto'].str.split(' ').str.get(1)  # Accede individualmente a los elementos
#df_productos['Producto'].str.split(' ').str[1]      # de la lista producida

# df_productos['Producto'].str.split(' ', expand=True) # En lugar de una lista crea columnas

`Series.str.extract()` acepta como parámetro una expresión regular y permite extraer el contenido de uno o más grupos de captura del primer emparejamiento. Solo extrae el primer emparejamiento.

In [ ]:
regex = (r'(?P<Producto>[^ ]+) '
         r'(?P<Procedencia>[^ ]+)')

df_productos['Producto'].str.upper().str.extract(regex)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> `str.extractall()` obtiene todas las coincidencias y utiliza un índice adicional para distinguirlas. Para comprobar patrones, `str.contains()` busca en cualquier parte, `str.match()` comprueba el comienzo y `str.fullmatch()` comprueba la cadena completa.
</div>

---
## <font color="#5E5E5E"> 3. Operaciones de transformación sobre `Series`</font>



Las funciones que operan sobre cada elemento de la `Series` de manera independiente, y devuelven una nueva `Series` de índice y tamaño similar, con el resultado de la operación sobre cada elemento original, se denominan **de transformación**. En cierto modo, algunas de las operaciones anteriores (sumar un escalar a una `Series`, convertir una `Series` de `string` etc.) son transformaciones. Sin embargo, no siempre se puede operar de forma vectorizada sobre las `Series`, y hay que actuar sobre cada elemento de forma individual.

### `Series.map()`

El método `Series.map()` toma como argumento una función *que actúa a nivel de elemento*,  o una tabla de correspondencias en forma de diccionario o `Series`, y devuelve una nueva `Series` con el mismo índice, y los valores transformados.  En el caso de recibir una función como argumento, la aplica sobre cada elemento de la `Series`. En el caso de recibir una tabla de correspondencia, las claves de ésta corresponden a los valores de los elementos de la `Series`, que son sustituidos por el correspondiente valor.
<br>



In [ ]:
# Transforma los valores mediante una función.
def discretize(elemento):
    if pd.isna(elemento):
        return pd.NA    
    elif elemento>=40:
        return 'Caro'
    elif elemento<=20:
        return 'Barato'
    else:
        return 'Justo'
    
df_productos['Venta'].map(discretize)

En el caso de recibir un diccionario, sustituye cada elemento (clave) por la entrada correspondiente al mismo (valor) en el diccionario.

In [ ]:
# Transforma los valores mediante un diccionario
cp_loc = {'02001':'Albacete', '02003':'Villarrobledo', '02004':'Almansa' }

df_productos['C.P'].map(cp_loc)

### `Series.apply()`

El método `Series.apply(función)` toma como argumento una función de transformación y, ***en el caso de los objetos de tipo `Series`***, tiene una funcionalidad equivalente a  `Series.map(función)`.  

In [ ]:
# Transforma los valores mediante una función.
def discretize(elemento):
    if elemento>40:
        return 'Caro'
    elif elemento<20:
        return 'Barato'
    else:
        return 'Justo'
    
df_productos['Venta'].apply(discretize)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> La diferencia principal entre `Series.apply()` y `Series.map()` es que `Series.apply()` **solamente** acepta funciones, que pueden ser también vectorizadas y, en ese caso, se aplican a la `Series`completa.
</div>

En ambos casos, se suelen utilizar (por legibilidad muchas veces) funciones `lambda`.

In [ ]:
df_productos['Venta'].map(lambda precio: 'Caro' if precio>30 else 'Barato')
#df_productos['Venta'].apply(lambda precio: 'Caro' if precio>30 else 'Barato')

### `Series.transform()`

`Series.transform()` acepta un argumento que puede ser una función o una lista de funciones de transformación. En el primer caso la funcionalidad es similar a la de `Series.map()` o `Series.apply()`, por lo que devuelve una `Series` con la transformación de los datos. 



In [ ]:
def iva(precio):
    return precio*1.21

df_productos['Venta'].transform(iva)

En el segundo, devuelve un `DataFrame` en el que cada columna corresponde a la aplicación de una función de la lista.

In [ ]:
df_productos['Venta'].transform([lambda el: el+1, iva])

Hay varias formas de cambiar los nombres de las columnas resultantes. Una es asignar un nuevo índice. La más sencilla consiste en utilizar un diccionario.

In [ ]:
df_productos['Venta'].transform({'Suma uno': lambda el:el+1, 'IVA': iva})

También se pueden establecer posteriormente las etiquetas del índice.

In [ ]:
df_productos['Venta'].transform([lambda el: el+1, iva]).set_axis(['Suma uno','IVA'], axis=1)

<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
**Nota recordatoria**: las funciones que se pasan como argumento a estos tres métodos se denominan **de transformación**, y operan a nivel de elemento. (Es algo más complejo que esto, pero el uso con `Series` va a ser casi siempre el que se ha visto aquí). 


</div>

---
## <font color="#5E5E5E"> 4. Operaciones de transformación sobre `DataFrames`</font>


### `Dataframe.map()`

El método `DataFrame.map(función)` recibe como argumento una `función` de transformación, y la aplica a cada elemento del `DataFrame`. 

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>  A diferencia de `Series.map()`, **solamente** acepta funciones.
</div>

In [ ]:
# Selecciona las columnas numéricas del DataFrame, y las convierte en un String
df_productos.select_dtypes(include=np.number).map(lambda ele: '{:.3f} eur'.format(ele))

### Transformación mediante `Dataframe.apply()`

El método `DataFrame.apply(función)` recibe como argumento una `función` que actúa **sobre objetos de tipo `Series`**. El parámetro `axis` determina qué índice se conserva. Con `axis=0\axis='index'`, `DataFrame.apply()` pasa cada **columna** como `Series` a la función. Con `axis=1/axis='columns'`, pasa cada **fila** como `Series`.


<div class="alert alert-block alert-warning">
    
<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
En realidad,  el resultado depende de lo que la función devuelva; no se limita a transformaciones elemento a elemento. Cuando la `función` opera de manera individual sobre cada elemento de la `Serie`, permite hacer *algunas* operaciones de transformación (más tarde se verá otro uso de este método).
</div>


In [ ]:
def descuento(serie):                     # Calcula un descuento
    return serie*0.95

df_productos[['Compra','Venta']].apply(descuento, axis=0)  # Aplica la función descuento a cada columna
#df_productos[['Compra','Venta']].apply(descuento, axis=1) # Aplica la función descuento a cada fila

Los resultados obtenidos con `axis=0` y `axis=1` son similares porque **este** uso de `DataFrame.apply()`, con una `función` de transformación, es similar a `DataFrame.map()`.

In [ ]:
df_productos[['Compra','Venta']].map(descuento)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>  En este escenario (transformaciones) cada método ofrece ventaja:
* `DataFrame.map(función)`: La función opera directamente sobre *cada elemento*. Es más conciso, e igual de eficiente, para funciones que llevan a cabo operaciones **no vectorizadas**. 
* `DataFrame.apply(función)`: La función opera sobre una `Series`. Es más eficiente para funciones que implementan **operaciones vectorizadas**. 
</div>

### `DataFrame.transform()`

`DataFrame.transform()` es parecida a `DataFrame.map()`. Lleva a cabo una transformación de los elementos de un `DataFrame`. Permite especificar, mediante un diccionario, qué filas/columnas se transforman, y también aplicar varias transformaciones a cada fila/columna. El parámetro `axis` permite indicar si las claves del diccionario corresponden a filas o a columnas. Como resultado, devuelve una estructura que contiene todas las transformaciones llevadas a cabo  [**( + inf )**](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html).

In [ ]:
df_productos

In [ ]:
df_productos.transform({'Venta': lambda p: f"{p*1.21:.1f} euros", 
                        'Localidad':str.upper})

---
## <font color="#5E5E5E">  <i class="fa fa-plus-circle" aria-hidden="true" style="color:#5E5E5E"></i> Eficiencia</font>


En el siguiente ejemplo se compara el tiempo de ejecución de una suma de una `Series` de 5000 elementos con un escalar mediante un bucle, primero, y mediante vectorización, después. 

In [ ]:
%%timeit -n 10 serie = pd.Series(np.random.randint(0,1000,5000))
for indice, valor in serie.items():
    serie.loc[indice]= valor+2

In [ ]:
%%timeit -n 10 serie = pd.Series(np.random.randint(0,1000,5000))
serie+=2

En la medida de lo posible, es conveniente utilizar operaciones vectorizadas directamente en lugar de métodos como `Series.map()` o `Series.apply()`, aunque estos dos últimos son mucho más eficientes que la iteración.

In [ ]:
%%timeit -n 10 serie = pd.Series(np.random.randint(0,1000,5000))
serie.map(lambda elem: elem+2)

In [ ]:
%%timeit -n 10 serie = pd.Series(np.random.randint(0,1000,5000))
serie.apply(lambda elem: elem+2)

Cuando la transformación se lleva a cabo en los elementos de un `DataFrame`, se da una diferencia de rendimiento entre `DataFrame.map(función)` y `DataFrame.apply(función)` *cuando la función es vectorizada*, ya que en el primer caso se llama para cada elemento, y en el segundo para cada fila/columna (`Series`).

In [ ]:
df = pd.DataFrame({'A':np.random.randint(0,1000,10000), 'B':np.random.randint(0,1000,10000)})

In [ ]:
%%timeit -n 10
df.map(lambda elem: elem+2)

In [ ]:
%%timeit -n 10
df.apply(lambda elem: elem+2, axis=0)

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>